# 07 - wav2vec 2.0 Fine-tuning (Colab T4 GPU)

- **輸入**：raw waveform `.wav` (16kHz mono) — 從 `data/processed/audio_16k/`
- **模型**：`facebook/wav2vec2-base`
  - 凍結 feature_encoder + 前 8 層 transformer
  - 只訓練最後 4 層 + classification head
- **評估**：StratifiedGroupKFold (K=5, group=speaker_id)
- **訓練**：batch=4 × grad_accum=8 = effective 32, epochs=20, patience=5
- **精度**：fp16 mixed precision
- **加速**：音訊檔先複製到 Colab 本地暫存，避免 Drive I/O 瓶頸

In [ ]:
# === 安裝相依套件 ===
!pip install -q transformers datasets accelerate peft librosa soundfile

In [ ]:
# === 路徑設定（支援 Colab 與本地執行）===
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SER-Project')
except (ImportError, ModuleNotFoundError):
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

sys.path.append(str(PROJECT_ROOT / 'src'))

AUDIO_DIR = PROJECT_ROOT / 'data' / 'processed' / 'audio_16k'
CKPT_DIR = PROJECT_ROOT / 'models' / 'checkpoints' / 'wav2vec'
RESULTS_DIR = PROJECT_ROOT / 'results'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
import torch
print(f'Device: {"GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === 複製音訊到 Colab 本地暫存（避免 Drive I/O 瓶頸） ===
import shutil

LOCAL_AUDIO = Path('/content/temp_audio')

if LOCAL_AUDIO.exists() and len(list(LOCAL_AUDIO.glob('*.wav'))) > 11000:
    print(f'Local cache exists: {len(list(LOCAL_AUDIO.glob("*.wav"))):,} files')
else:
    print('Copying audio files to local storage (one-time, ~1-2 min)...')
    LOCAL_AUDIO.mkdir(exist_ok=True)
    !cp "{AUDIO_DIR}"/*.wav /content/temp_audio/
    n_files = len(list(LOCAL_AUDIO.glob('*.wav')))
    print(f'Done: {n_files:,} files copied to {LOCAL_AUDIO}')

In [ ]:
# === Imports ===
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm

from models import EarlyStopping

# === 超參數 ===
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8    # effective batch = 32
EPOCHS = 20
LR = 2e-5
PATIENCE = 5
N_SPLITS = 5
RANDOM_STATE = 42
MAX_GRAD_NORM = 1.0     # gradient clipping
MAX_LENGTH_SEC = 3.0
TARGET_SR = 16000
MAX_LENGTH_SAMPLES = int(MAX_LENGTH_SEC * TARGET_SR)  # 48000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 可重現性 ===
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Device: {DEVICE}')

In [ ]:
# === Dataset ===
class AudioDataset(Dataset):
    """Raw waveform dataset: 載入 .wav，pad/truncate 至固定長度。"""

    def __init__(self, paths: list[str], labels: np.ndarray,
                 feature_extractor: Wav2Vec2FeatureExtractor,
                 max_length: int = MAX_LENGTH_SAMPLES):
        self.paths = paths
        self.labels = labels
        self.fe = feature_extractor
        self.max_length = max_length

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio, _ = librosa.load(self.paths[idx], sr=TARGET_SR)

        # pad/truncate（取中間段）
        if len(audio) > self.max_length:
            start = (len(audio) - self.max_length) // 2
            audio = audio[start:start + self.max_length]
        elif len(audio) < self.max_length:
            pad_total = self.max_length - len(audio)
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            audio = np.pad(audio, (pad_left, pad_right))

        inputs = self.fe(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=False)
        x = inputs['input_values'].squeeze(0)  # (48000,)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# === 載入 metadata ===
df = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
le = LabelEncoder()
y_all = le.fit_transform(df['emotion'].values)
groups = df['speaker_id'].values
classes = le.classes_.tolist()

# 使用本地暫存路徑（檔名相同，只是目錄不同）
audio_paths = [str(LOCAL_AUDIO / Path(p).name) for p in df['processed_path']]

# wav2vec2 feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')

print(f'Samples: {len(df):,}, Classes: {classes}')
print(f'Speakers: {df["speaker_id"].nunique()}')
print(f'Audio source: {LOCAL_AUDIO} (local cache)')

In [ ]:
# === 模型建構與凍結 ===

def create_wav2vec_model(num_labels: int) -> Wav2Vec2ForSequenceClassification:
    """建構 wav2vec2 分類模型，凍結 feature_encoder + 前 8 層 transformer。"""
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        'facebook/wav2vec2-base',
        num_labels=num_labels,
        classifier_proj_size=256,
    )
    model.freeze_feature_encoder()
    for i in range(8):
        for param in model.wav2vec2.encoder.layers[i].parameters():
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {total:,} total, {trainable:,} trainable '
          f'({trainable/total*100:.1f}%)')
    return model

# 測試建構
_test = create_wav2vec_model(len(classes))
del _test
torch.cuda.empty_cache()

In [ ]:
# === 訓練 / 評估函式（外部 loss + gradient accumulation + clipping）===

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, grad_accum_steps):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for step, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(X)           # 不傳 labels，自行算 loss
            logits = outputs.logits
            loss = criterion(logits, y)
            scaled_loss = loss / grad_accum_steps
        scaler.scale(scaled_loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(X)
            logits = outputs.logits
            loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    n = len(all_labels)
    preds = np.array(all_preds)
    labels = np.array(all_labels)
    return (
        total_loss / n,
        accuracy_score(labels, preds),
        f1_score(labels, preds, average='weighted'),
        f1_score(labels, preds, average='macro'),
        preds,
        labels,
    )

In [ ]:
# === 5-Fold 訓練（支援斷點續跑：每 fold 結果獨立存檔） ===
FOLD_RESULTS_DIR = RESULTS_DIR / 'wav2vec_folds'
FOLD_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
all_results = {'folds': [], 'confusion_matrices': [], 'classification_reports': [], 'classes': classes}

for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(audio_paths, y_all, groups)):
    fold_result_path = FOLD_RESULTS_DIR / f'wav2vec_fold_{fold_idx+1}_result.json'

    # === 斷點續跑：若該 fold 結果已存在，載入並跳過 ===
    if fold_result_path.exists():
        with open(fold_result_path, encoding='utf-8') as f:
            saved = json.load(f)
        all_results['folds'].append(saved['fold_result'])
        all_results['confusion_matrices'].append(saved['confusion_matrix'])
        all_results['classification_reports'].append(saved['classification_report'])
        print(f'[SKIP] Fold {fold_idx+1} already done '
              f'(acc={saved["fold_result"]["accuracy"]:.4f}, '
              f'F1(m)={saved["fold_result"]["f1_macro"]:.4f})')
        continue

    print(f'\n{"="*60}')
    print(f'  Fold {fold_idx + 1}/{N_SPLITS}')
    print(f'{"="*60}')

    # 資料集
    train_paths = [audio_paths[i] for i in train_idx]
    test_paths = [audio_paths[i] for i in test_idx]
    train_ds = AudioDataset(train_paths, y_all[train_idx], feature_extractor)
    test_ds = AudioDataset(test_paths, y_all[test_idx], feature_extractor)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Class weight（根據訓練集類別分布計算逆頻率權重）
    cw = compute_class_weight('balanced', classes=np.arange(len(classes)), y=y_all[train_idx])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
    print(f'  Class weights: {dict(zip(classes, [f"{w:.3f}" for w in cw]))}')

    # 模型
    model = create_wav2vec_model(len(classes)).to(DEVICE)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler()
    early_stopping = EarlyStopping(patience=PATIENCE)

    best_val_loss = float('inf')
    t_start = time.time()

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, DEVICE, GRAD_ACCUM_STEPS)
        val_loss, val_acc, val_f1w, val_f1m, _, _ = evaluate(
            model, test_loader, criterion, DEVICE)
        scheduler.step(val_loss)
        early_stopping(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), CKPT_DIR / f'wav2vec_fold{fold_idx+1}_best.pt')

        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch+1:>2d}: train_loss={train_loss:.4f}, '
              f'val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, '
              f'F1(w)={val_f1w:.4f}, F1(m)={val_f1m:.4f}, lr={lr_now:.1e}')

        if early_stopping.should_stop:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # 載入最佳權重
    model.load_state_dict(torch.load(CKPT_DIR / f'wav2vec_fold{fold_idx+1}_best.pt', weights_only=True))
    val_loss, val_acc, val_f1w, val_f1m, preds, labels = evaluate(
        model, test_loader, criterion, DEVICE)
    elapsed = time.time() - t_start

    cm = confusion_matrix(labels, preds).tolist()
    report = classification_report(labels, preds, target_names=classes, output_dict=True)

    fold_result = {
        'fold': fold_idx + 1,
        'accuracy': val_acc,
        'f1_weighted': val_f1w,
        'f1_macro': val_f1m,
        'best_val_loss': best_val_loss,
        'stopped_epoch': epoch + 1,
        'time_sec': round(elapsed, 1),
        'test_samples': len(test_idx),
        'train_samples': len(train_idx),
    }
    all_results['folds'].append(fold_result)
    all_results['confusion_matrices'].append(cm)
    all_results['classification_reports'].append(report)

    # === 立即存檔：該 fold 結果寫入獨立 JSON ===
    fold_save = {
        'fold_result': fold_result,
        'confusion_matrix': cm,
        'classification_report': report,
    }
    with open(fold_result_path, 'w', encoding='utf-8') as f:
        json.dump(fold_save, f, indent=2, ensure_ascii=False)

    print(f'\n  [Fold {fold_idx+1} Best] acc={val_acc:.4f}, F1(w)={val_f1w:.4f}, '
          f'F1(macro)={val_f1m:.4f}, time={elapsed:.0f}s')
    print(f'  [SAVED] {fold_result_path.name}')

    # 釋放記憶體
    del model, optimizer, scaler, criterion
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('  All folds done!')
print(f'{"="*60}')

In [ ]:
# === 結果匯總 + 儲存 ===
metrics_df = pd.DataFrame(all_results['folds'])

summary = {
    'accuracy_mean': metrics_df['accuracy'].mean(),
    'accuracy_std': metrics_df['accuracy'].std(),
    'f1_weighted_mean': metrics_df['f1_weighted'].mean(),
    'f1_weighted_std': metrics_df['f1_weighted'].std(),
    'f1_macro_mean': metrics_df['f1_macro'].mean(),
    'f1_macro_std': metrics_df['f1_macro'].std(),
}
all_results['summary'] = summary

with open(RESULTS_DIR / 'wav2vec_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print('wav2vec2 5-Fold Results:')
print(f'  Accuracy:    {summary["accuracy_mean"]:.4f} +/- {summary["accuracy_std"]:.4f}')
print(f'  F1 (weighted): {summary["f1_weighted_mean"]:.4f} +/- {summary["f1_weighted_std"]:.4f}')
print(f'  F1 (macro):    {summary["f1_macro_mean"]:.4f} +/- {summary["f1_macro_std"]:.4f}')
print(f'\nResults saved to {RESULTS_DIR / "wav2vec_results.json"}')
print(f'Checkpoints saved to {CKPT_DIR}')

In [ ]:
# === Per-fold 表格 ===
metrics_df[['fold', 'accuracy', 'f1_weighted', 'f1_macro', 'stopped_epoch', 'time_sec']]